# 02 - Feature Engineering (Credit Card Transactions)

**Phase goal:** Create and transform features from the cleaned Parquet data produced in Phase 01 to improve the quality and effectiveness of downstream analytical or machine-learning tasks.

## Features engineered in this notebook

| Group | Features |
|---|---|
| Temporal | `tx_hour`, `tx_day_of_week`, `tx_month`, `tx_is_weekend`, `tx_time_of_day` |
| Amount | `amount_log1p`, `amount_zscore`, `amount_bin` |
| Geo-distance | `merch_distance_km` (haversine between cardholder and merchant lat/long) |
| Card velocity | `card_tx_count_1h`, `card_amount_sum_1h` (rolling 1-hour window per card) |
| Cardholder | `cardholder_age`, `gender_encoded` |
| Merchant | `merchant_fraud_rate` (historical mean fraud label per merchant) |
| Category | `category_encoded` (StringIndexer) |

**Input:** `data/processed/transactions_parquet`  
**Output:** `data/processed/features_parquet`

In [ ]:
# Install dependencies if needed (Colab)
# !pip install -r requirements.txt

from pathlib import Path
import sys
import math

cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline import create_spark_session
from pyspark.sql import functions as F, Window
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import StringIndexer, Bucketizer
from pyspark.ml import Pipeline as MLPipeline

processed_dir = project_root / "data" / "processed"
input_parquet  = processed_dir / "transactions_parquet"
output_parquet = processed_dir / "features_parquet"

spark = create_spark_session(app_name="CreditCardFeatureEngineering")
print(f"Input  : {input_parquet}")
print(f"Output : {output_parquet}")

## 1. Load cleaned data

In [ ]:
df = spark.read.parquet(str(input_parquet))
print(f"Loaded {df.count():,} rows | {len(df.columns)} columns")
df.printSchema()
df.show(5, truncate=False)

## 2. Temporal features

Extract hour, day-of-week, month, weekend flag, and a bucketed time-of-day label from `transaction_ts`.

In [ ]:
df = df.withColumn("tx_hour",        F.hour("transaction_ts")) \
       .withColumn("tx_day_of_week", F.dayofweek("transaction_ts")) \
       .withColumn("tx_month",       F.month("transaction_ts")) \
       .withColumn("tx_is_weekend",  (F.dayofweek("transaction_ts").isin(1, 7)).cast("int")) \
       .withColumn(
           "tx_time_of_day",
           F.when(F.col("tx_hour").between(0,  5),  "night")
            .when(F.col("tx_hour").between(6, 11),  "morning")
            .when(F.col("tx_hour").between(12, 17), "afternoon")
            .otherwise("evening")
       )

df.select("transaction_id", "transaction_ts",
          "tx_hour", "tx_day_of_week", "tx_month",
          "tx_is_weekend", "tx_time_of_day").show(10)

## 3. Amount features

- **`amount_log1p`** – log1p transformation to reduce skew.
- **`amount_zscore`** – z-score normalisation (mean/std over entire dataset).
- **`amount_bin`** – bucketed label: `micro` (<10), `small` (10–100), `medium` (100–500), `large` (≥500).

In [ ]:
# log1p
df = df.withColumn("amount_log1p", F.log1p(F.col("amount")))

# z-score
amt_stats = df.select(
    F.mean("amount").alias("mean_amt"),
    F.stddev("amount").alias("std_amt")
).first()
mean_amt = amt_stats["mean_amt"]
std_amt  = amt_stats["std_amt"]
print(f"Amount  mean={mean_amt:.2f}  std={std_amt:.2f}")

df = df.withColumn(
    "amount_zscore",
    ((F.col("amount") - F.lit(mean_amt)) / F.lit(std_amt)).cast(DoubleType())
)

# bin
df = df.withColumn(
    "amount_bin",
    F.when(F.col("amount") < 10,   "micro")
     .when(F.col("amount") < 100,  "small")
     .when(F.col("amount") < 500,  "medium")
     .otherwise("large")
)

df.select("transaction_id", "amount", "amount_log1p", "amount_zscore", "amount_bin").show(10)

## 4. Geo-distance feature

Haversine distance (km) between cardholder home location (`lat`, `long`) and merchant location (`merch_lat`, `merch_long`).  
Large distances between home and merchant can be a strong fraud indicator.

In [ ]:
# Haversine formula as a Spark UDF
@F.udf(returnType=DoubleType())
def haversine_km(lat1, lon1, lat2, lon2):
    if any(v is None for v in [lat1, lon1, lat2, lon2]):
        return None
    R = 6371.0  # Earth radius in km
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

geo_cols = {"lat", "long", "merch_lat", "merch_long"}
if geo_cols.issubset(set(df.columns)):
    df = df.withColumn(
        "merch_distance_km",
        haversine_km(F.col("lat"), F.col("long"), F.col("merch_lat"), F.col("merch_long"))
    )
    df.select("transaction_id", "lat", "long", "merch_lat", "merch_long", "merch_distance_km").show(10)
else:
    print(f"Skipping geo-distance: missing columns. Available: {df.columns}")

## 5. Card velocity features (rolling 1-hour window)

Per-card rolling aggregations look back 1 hour before each transaction:
- **`card_tx_count_1h`** – number of transactions on the card in the prior hour.
- **`card_amount_sum_1h`** – total spend on the card in the prior hour.

High velocity is a classic fraud signal.

In [ ]:
# Convert transaction_ts to unix seconds for range-based window
df = df.withColumn("tx_unix", F.unix_timestamp("transaction_ts"))

one_hour = 3600  # seconds

if "card_id" in df.columns:
    w_vel = (
        Window.partitionBy("card_id")
              .orderBy("tx_unix")
              .rangeBetween(-one_hour, -1)  # exclude current row
    )
    df = df.withColumn("card_tx_count_1h",  F.count("transaction_id").over(w_vel)) \
           .withColumn("card_amount_sum_1h", F.sum("amount").over(w_vel))

    df.select("transaction_id", "card_id", "transaction_ts",
              "amount", "card_tx_count_1h", "card_amount_sum_1h").show(15)
else:
    print("Skipping velocity: 'card_id' column not found.")

## 6. Cardholder features

- **`cardholder_age`** – derived from `dob` relative to the transaction date.
- **`gender_encoded`** – binary encode gender (M=1, F=0).

In [ ]:
if "dob" in df.columns:
    df = df.withColumn("dob_ts", F.to_date(F.col("dob"), "yyyy-MM-dd")) \
           .withColumn(
               "cardholder_age",
               F.floor(F.datediff(F.col("transaction_ts").cast("date"), F.col("dob_ts")) / 365.25)
           ).drop("dob_ts")
    print("Cardholder age sample:")
    df.select("transaction_id", "dob", "cardholder_age").show(10)
else:
    print("Skipping age: 'dob' column not found.")

if "gender" in df.columns:
    df = df.withColumn(
        "gender_encoded",
        F.when(F.upper(F.col("gender")) == "M", 1).otherwise(0).cast("int")
    )
    print("Gender encoding distribution:")
    df.groupBy("gender", "gender_encoded").count().show()
else:
    print("Skipping gender: 'gender' column not found.")

## 7. Merchant historical fraud rate

Mean `is_fraud` per merchant — a target-encoded feature.  
Higher values indicate merchants historically associated with more fraud.

In [ ]:
if "merchant" in df.columns and "is_fraud" in df.columns:
    merchant_fraud_rate = df.groupBy("merchant").agg(
        F.mean("is_fraud").alias("merchant_fraud_rate"),
        F.count("*").alias("merchant_tx_count")
    )
    print("Top merchants by fraud rate:")
    merchant_fraud_rate.orderBy(F.col("merchant_fraud_rate").desc()).show(10)

    df = df.join(merchant_fraud_rate.select("merchant", "merchant_fraud_rate"),
                 on="merchant", how="left")
else:
    print("Skipping merchant fraud rate: required columns not found.")

## 8. Category encoding (StringIndexer)

Encode the `category` string column to a numeric index using Spark ML's `StringIndexer`, ordered by frequency.

In [ ]:
if "category" in df.columns:
    indexer = StringIndexer(
        inputCol="category",
        outputCol="category_encoded",
        handleInvalid="keep"
    )
    indexer_model = indexer.fit(df)
    df = indexer_model.transform(df)

    print("Category index mapping (label -> index):")
    for idx, label in enumerate(indexer_model.labels):
        print(f"  {idx:>2}: {label}")

    df.select("category", "category_encoded").distinct().orderBy("category_encoded").show(20)
else:
    print("Skipping category encoding: 'category' column not found.")

## 9. Final schema & quality check

In [ ]:
engineered_cols = [
    "tx_hour", "tx_day_of_week", "tx_month", "tx_is_weekend", "tx_time_of_day",
    "amount_log1p", "amount_zscore", "amount_bin",
    "merch_distance_km",
    "card_tx_count_1h", "card_amount_sum_1h",
    "cardholder_age", "gender_encoded",
    "merchant_fraud_rate",
    "category_encoded",
]

present = [c for c in engineered_cols if c in df.columns]
missing = [c for c in engineered_cols if c not in df.columns]

print(f"Engineered features present ({len(present)}): {present}")
if missing:
    print(f"Skipped (source columns absent): {missing}")

print(f"\nTotal columns : {len(df.columns)}")
print(f"Total rows    : {df.count():,}")
df.select(present[:8]).describe().show()

## 10. Write features Parquet

In [ ]:
df.write.mode("overwrite").parquet(str(output_parquet))
print(f"Features Parquet written to: {output_parquet}")

# Verify round-trip
verify = spark.read.parquet(str(output_parquet))
print(f"Verified read-back: {verify.count():,} rows | {len(verify.columns)} columns")

spark.stop()
print("Spark session stopped.")